> **Portfolio summary:** Credit-risk classification with a PyTorch multilayer perceptron, from preprocessing to evaluation.
>
> [Project README](README.md) · [Back to the portfolio](../README.md)

This notebook demonstrates a complete machine learning workflow, from data loading and preprocessing to model training and evaluation. We'll be working with a credit risk dataset, where the goal is to predict creditworthiness. The process will involve exploratory data analysis, feature engineering (including one-hot encoding and scaling), building a Multilayer Perceptron (MLP) model using PyTorch, and finally, evaluating its performance. This structured approach highlights best practices in developing a predictive model


In [ ]:
# Evaluate the model
model.eval() # Set the model to evaluation mode
with torch.no_grad(): # Disable gradient calculation during evaluation
    correct = 0
    total = 0
    for inputs, labels in test_loader:
        outputs = model(inputs)
        predicted = (outputs > 0.5).float() # Convert probabilities to binary predictions (0 or 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = correct / total
    print(f'Accuracy on the test set: {accuracy:.4f}')

Accuracy on the test set: 0.7500

In [ ]:
import torch.optim as optim

# Define Loss function and Optimizer
criterion = nn.BCELoss() # Binary Cross-Entropy Loss for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam optimizer with a learning rate

# Training the model
num_epochs = 50 # As specified in the problem

for epoch in range(num_epochs):
    model.train() # Set the model to training mode
    for inputs, labels in train_loader:
        optimizer.zero_grad() # Zero the gradients
        outputs = model(inputs) # Forward pass
        loss = criterion(outputs, labels) # Compute loss
        loss.backward() # Backward pass
        optimizer.step() # Update weights

    # Print loss every 10 epochs
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("Training complete.")

Epoch [10/50], Loss: 0.3508
Epoch [20/50], Loss: 0.1813
Epoch [30/50], Loss: 0.0443
Epoch [40/50], Loss: 0.0075
Epoch [50/50], Loss: 0.0060
Training complete.

In [ ]:
import torch.nn as nn

# Define the MLP model
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size_1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size_2, output_size)
        self.sigmoid = nn.Sigmoid() # For binary classification

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        out = self.sigmoid(out)
        return out

# Instantiate the model
input_size = X_train_scaled.shape[1] # Number of features
hidden_size_1 = 64 # You can adjust this
hidden_size_2 = 32 # You can adjust this
output_size = 1 # Binary classification

model = MLP(input_size, hidden_size_1, hidden_size_2, output_size)

print(model)
print(f"Number of parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

MLP(
  (fc1): Linear(in_features=48, out_features=64, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Number of parameters: 5249

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert scaled DataFrames and target Series to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1) # unsqueeze for BCELoss
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# Create TensorDatasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
batch_size = 32 # You can adjust this batch size
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")

# Display a sample batch (optional)
for features, labels in train_loader:
    print("Sample X_train_batch shape:", features.shape)
    print("Sample y_train_batch shape:", labels.shape)
    break

Number of training batches: 25
Number of testing batches: 7
Sample X_train_batch shape: torch.Size([32, 48])
Sample y_train_batch shape: torch.Size([32, 1])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Initialize the StandardScaler
scaler = StandardScaler()

# Apply StandardScaler to numerical features.
# It's important to fit the scaler only on the training data to prevent data leakage.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames for easier inspection (optional, but good for understanding)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Shape of X_train_scaled:", X_train_scaled.shape)
print("Shape of X_test_scaled:", X_test_scaled.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

display(X_train_scaled.head())

Shape of X_train_scaled: (800, 48)
Shape of X_test_scaled: (200, 48)
Shape of y_train: (800,)
Shape of y_test: (200,)

,Duration in month,Credit amount,Installment rate in percentage of disposable income,Present residence since,Age in years,Number of existing credits at this bank,Number of people being liable to provide maintenance for,Status of existing checking account_A12,Status of existing checking account_A13,Status of existing checking account_A14,...,Property_A124,Other installment plans_A142,Other installment plans_A143,Housing_A152,Housing_A153,Job_A172,Job_A173,Job_A174,Telephone_A192,foreign worker_A202
0,3.297082,1.199912,0.031196,1.044509,2.406187,1.017777,-0.409736,-0.604308,-0.255434,-0.822891,...,2.346074,-0.213896,0.468521,0.629413,-0.349313,-0.480384,0.764303,-0.436436,1.175139,-0.190445
1,-0.008051,-0.359630,-0.860109,-1.671440,-0.224364,-0.710931,-0.409736,-0.604308,3.914911,-0.822891,...,-0.426244,-0.213896,0.468521,-1.588782,-0.349313,-0.480384,0.764303,-0.436436,-0.850963,-0.190445
2,-1.279256,-0.733547,-0.860109,1.044509,1.266282,-0.710931,-0.409736,-0.604308,-0.255434,1.215228,...,-0.426244,-0.213896,0.468521,-1.588782,-0.349313,-0.480384,0.764303,-0.436436,-0.850963,-0.190445
3,-0.008051,0.567050,-1.751413,1.044509,-0.575104,1.017777,-0.409736,-0.604308,-0.255434,1.215228,...,-0.426244,-0.213896,-2.134375,0.629413,-0.349313,-0.480384,0.764303,-0.436436,1.175139,-0.190445
4,-0.770774,-0.854388,0.922500,-0.766124,-1.276585,-0.710931,-0.409736,-0.604308,-0.255434,1.215228,...,-0.426244,-0.213896,0.468521,0.629413,-0.349313,-0.480384,0.764303,-0.436436,-0.850963,-0.190445


In [ ]:
# Map 'Creditability' to 0 and 1
df['Creditability'] = df['Creditability'].replace({1: 1, 2: 0})

# Separate features (X) and target (y)
X = df.drop('Creditability', axis=1)
y = df['Creditability']

# Identify categorical columns (object type) for one-hot encoding
categorical_cols = X.select_dtypes(include=['object']).columns

# Apply one-hot encoding to categorical columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Display the first few rows of the encoded DataFrame and its shape
print("Shape of X_encoded:", X_encoded.shape)
display(X_encoded.head())

Shape of X_encoded: (1000, 48)

,Duration in month,Credit amount,Installment rate in percentage of disposable income,Present residence since,Age in years,Number of existing credits at this bank,Number of people being liable to provide maintenance for,Status of existing checking account_A12,Status of existing checking account_A13,Status of existing checking account_A14,...,Property_A124,Other installment plans_A142,Other installment plans_A143,Housing_A152,Housing_A153,Job_A172,Job_A173,Job_A174,Telephone_A192,foreign worker_A202
0,6,1169,4,4,67,2,1,False,False,False,...,False,False,True,True,False,False,True,False,True,False
1,48,5951,2,2,22,1,1,True,False,False,...,False,False,True,True,False,False,True,False,False,False
2,12,2096,2,3,49,1,2,False,False,True,...,False,False,True,True,False,True,False,False,False,False
3,42,7882,2,4,45,1,2,False,False,False,...,False,False,True,False,True,False,True,False,False,False
4,24,4870,3,4,53,2,2,False,False,False,...,True,False,True,False,True,False,True,False,False,False


In [ ]:
import pandas as pd

# URL of the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

# Column names provided in the hint
column_names = [
    "Status of existing checking account", "Duration in month", "Credit history",
    "Purpose", "Credit amount", "Savings account/bonds", "Present employment since",
    "Installment rate in percentage of disposable income", "Personal status and sex",
    "Other debtors / guarantors", "Present residence since", "Property",
    "Age in years", "Other installment plans", "Housing",
    "Number of existing credits at this bank", "Job",
    "Number of people being liable to provide maintenance for", "Telephone",
    "foreign worker", "Creditability"
]

# Load the dataset, using space as a separator as specified, and assign column names
df = pd.read_csv(url, sep=' ', names=column_names)

# Display the first 5 rows of the DataFrame to verify loading
display(df.head())

,Status of existing checking account,Duration in month,Credit history,Purpose,Credit amount,Savings account/bonds,Present employment since,Installment rate in percentage of disposable income,Personal status and sex,Other debtors / guarantors,...,Property,Age in years,Other installment plans,Housing,Number of existing credits at this bank,Job,Number of people being liable to provide maintenance for,Telephone,foreign worker,Creditability
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2
